# Evaluation
## Credit Risk Scorecard Project

CRISP-DM Phase 5. This notebook evaluates both models using
credit-risk-specific metrics and connects them to the business decision:

1. KS Statistic and Gini coefficient
2. Probability calibration
3. Cost-based threshold selection
4. Expected profit per 1,000 applicants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

In [ ]:
app = pd.read_csv('data/processed/app_clean.csv')
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app['TARGET']
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X.columns]

# Same splits as notebook 04
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval)

# Impute with TRAIN medians only
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Protected attribute removed from the deployed model
gender_cols = [c for c in X.columns if 'GENDER' in c.upper()]
feats = [c for c in X.columns if c not in gender_cols]
print("Removed:", gender_cols, "| features used:", len(feats))

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
params = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.05,
          'num_leaves': 31, 'scale_pos_weight': neg/pos,
          'random_state': 42, 'verbose': -1}

def fit(cols):
    d = lgb.Dataset(X_train[cols], label=y_train)
    v = lgb.Dataset(X_val[cols], label=y_val, reference=d)
    return lgb.train(params, d, num_boost_round=500, valid_sets=[v],
                     callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

model = fit(feats)                  # gender-free (the model we use)
model_full = fit(list(X.columns))   # with gender (fairness comparison only)

raw_test = model.predict(X_test[feats], num_iteration=model.best_iteration)
raw_val = model.predict(X_val[feats], num_iteration=model.best_iteration)
test_auc = roc_auc_score(y_test, raw_test)
auc_full = roc_auc_score(y_test, model_full.predict(X_test[list(X.columns)],
                         num_iteration=model_full.best_iteration))
print(f"Best iteration: {model.best_iteration}")
print(f"Gender-free test AUC: {test_auc:.4f} | with gender: {auc_full:.4f}")

In [ ]:
def calculate_ks(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return max(tpr - fpr)

ks = calculate_ks(y_test, raw_test)
gini = 2 * test_auc - 1
print(f"AUC: {test_auc:.4f} | KS: {ks:.4f} | Gini: {gini:.4f}")

In [ ]:
# Calibrator fitted on VALIDATION. Bounds avoid exact 0 and 1 probabilities.
iso = IsotonicRegression(out_of_bounds='clip', y_min=0.001, y_max=0.999)
iso.fit(raw_val, y_val)

# Test set -> half A (choose threshold) and half B (report profit)
raw_A, raw_B, yA, yB = train_test_split(
    raw_test, y_test.values, test_size=0.5, random_state=42, stratify=y_test.values)
cal_A, cal_B = iso.predict(raw_A), iso.predict(raw_B)
print(f"Half A: {len(yA):,} | Half B: {len(yB):,}")

In [ ]:
t_raw, p_raw = calibration_curve(yB, raw_B, n_bins=10)
t_cal, p_cal = calibration_curve(yB, cal_B, n_bins=10)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(p_raw, t_raw, marker='o', color='crimson', label='Before (raw LightGBM)')
ax.plot(p_cal, t_cal, marker='s', color='green', label='After (isotonic)')
ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Actual Default Rate')
ax.set_title('Calibration: Before vs After (test half B)'); ax.legend()
plt.tight_layout(); plt.savefig('outputs/05_calibration_comparison.png', dpi=150); plt.show()

In [ ]:
PROFIT_GOOD, LOSS_BAD = 1000, 6000

def profit_table(y_true, p):
    rows = []
    for t in np.round(np.arange(0.01, 1.00, 0.01), 2):
        ok = p < t
        g, b = int((ok & (y_true == 0)).sum()), int((ok & (y_true == 1)).sum())
        rows.append({'threshold': t, 'approval_rate': ok.mean(),
                     'approved_good': g, 'approved_bad': b,
                     'profit': g * PROFIT_GOOD - b * LOSS_BAD})
    return pd.DataFrame(rows)

tbl_A, tbl_B = profit_table(yA, cal_A), profit_table(yB, cal_B)
best_t = tbl_A.loc[tbl_A['profit'].idxmax(), 'threshold']      # chosen on A
row_B = tbl_B[tbl_B['threshold'] == best_t].iloc[0]            # reported on B
breakeven = PROFIT_GOOD / (PROFIT_GOOD + LOSS_BAD)

profit_all_B = int((yB == 0).sum() * PROFIT_GOOD - (yB == 1).sum() * LOSS_BAD)
print(f"Threshold chosen on A: {best_t:.2f} (theoretical break-even {breakeven:.3f})")
print(f"Half B approval rate: {row_B['approval_rate']:.1%}")
print(f"Half B profit (model):        ${row_B['profit']:,.0f}")
print(f"Half B profit (approve all):  ${profit_all_B:,}")
print(f"Profit per applicant (model): ${row_B['profit']/len(yB):,.0f}")
print(f"Uplift vs approve-all per applicant: ${(row_B['profit']-profit_all_B)/len(yB):,.0f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tbl_A['threshold'], tbl_A['profit'], label='Half A (used to choose)')
ax.plot(tbl_B['threshold'], tbl_B['profit'], label='Half B (reported)')
ax.axvline(best_t, color='crimson', ls='--', label=f'Chosen {best_t:.2f}')
ax.axvline(breakeven, color='orange', ls=':', label=f'Break-even {breakeven:.3f}')
ax.set_xlabel('Approval threshold'); ax.set_ylabel('Profit ($)'); ax.legend()
plt.tight_layout(); plt.savefig('outputs/05_profit_threshold.png', dpi=150); plt.show()

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Features', 'AUC', 'KS', 'Gini', 'Threshold', 'Profit per applicant'],
    'WOE Scorecard': ['12', '0.7345', '0.3506', '0.4691', 'N/A', 'N/A'],
    'LightGBM (no gender)': [str(len(feats)), f'{test_auc:.4f}', f'{ks:.4f}',
                             f'{gini:.4f}', f'{best_t:.2f}',
                             f"${row_B['profit']/len(yB):,.0f}"]})
print(summary.to_string(index=False))

### Summary & Recommendation

LightGBM (gender removed) outperforms the WOE scorecard across all ranking metrics
(AUC 0.768 vs 0.735, KS 0.403 vs 0.351, Gini 0.536 vs 0.469). Part of that gap comes
from LightGBM using 272 features against the scorecard's 12 (see notebook 04, which
compares both models on the same 12 features). The scorecard's advantage is
interpretability: every decision can be explained to an applicant or regulator.

Because LightGBM is trained with class weights, its raw scores are not true
probabilities. Isotonic regression, fitted on a held-out validation set, maps them
to calibrated probabilities that can support risk-based pricing and cost-based
decisions rather than ranking alone.

The approval threshold was chosen on one half of the test set (0.16, near the
theoretical break-even of 1,000 / 7,000 = 0.143) and profit was reported on the other
half, which played no part in that choice. On that held-out half the model approves
87.2% of applicants and earns $548 per applicant, against $435 per applicant if
everyone is approved, an uplift of $113 per applicant. These figures rest on assumed
flat costs (+$1,000 per good loan, -$6,000 per bad loan); see Limitations.

**Recommendation:** Deploy the calibrated, gender-free LightGBM for automated ranking
and approval decisions, while retaining the scorecard for cases requiring transparent,
explainable reasoning.

In [ ]:
print(f"AUC with gender:    {auc_full:.4f}")
print(f"AUC without gender: {test_auc:.4f}")
print(f"Difference:         {auc_full - test_auc:.4f}")

### Fairness Analysis: Removing the Protected Attribute

CODE_GENDER_F ranked #11 in the feature importance of the model that was allowed to
use gender, so the model does use gender to predict default. Using gender in credit
decisions is prohibited under fair-lending regulations in most jurisdictions.

Retraining without any gender feature reduced test AUC from 0.7700 to 0.7679, a drop
of about 0.002. That cost is small, so there is no meaningful performance
justification for keeping a protected attribute. The deployed model excludes it.

**Important caveat:** Removing the gender column is necessary but not sufficient.
The model may still discriminate indirectly through correlated proxy features such as
occupation or income type. A full fairness audit would require disparate impact
testing across protected groups, not just removal of the explicit attribute.